# Week 1 — Model Context Protocol: Setup & Architecture

> **Source notebook for** [`src/mcp_core/`](../src/mcp_core/).
> All implementations referenced below are checked into the repository.

## Learning objectives

By the end of this notebook you will be able to:

1. Articulate the JSON-RPC 2.0 envelope on which MCP rides, and explain why it was chosen.
2. Implement the MCP lifecycle (`initialize` → `tools/list` → `tools/call` → `shutdown`) by hand.
3. Define a tool with a JSON-Schema-validated input contract.
4. Expose a sandboxed local filesystem and a SQLite database as read-only MCP resources.
5. Drive your own server from a minimal client over stdio.


## 1. The protocol, formalized

The Model Context Protocol is a thin layer on top of **JSON-RPC 2.0**. Every message is one of three forms:

**Request**:
```json
{ "jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {} }
```

**Response** (success):
```json
{ "jsonrpc": "2.0", "id": 1, "result": { "tools": [ ... ] } }
```

**Response** (error):
```json
{ "jsonrpc": "2.0", "id": 1, "error": { "code": -32601, "message": "Method not found" } }
```

**Notification** — a request without `id`, expecting no response.

### Why JSON-RPC and not REST or gRPC?

| Property | JSON-RPC 2.0 | REST | gRPC |
|----------|--------------|------|------|
| Bidirectional over a single channel | ✓ | ✗ | ✓ |
| Transport-agnostic (stdio, ws, http) | ✓ | partially | ✗ (HTTP/2) |
| Schemaless wire format | ✓ | ✓ | ✗ (Protobuf) |
| Streaming responses | ✓ via notifications | needs SSE | ✓ |

MCP needs to ride on stdio (so a desktop LLM client can spawn the server as a child process) **and** over websockets (so a hosted model can connect over the network). JSON-RPC's transport-agnosticism is therefore the deciding factor.


## 2. Lifecycle and capability negotiation

A compliant exchange looks like this:

```
Client                                Server
  │                                     │
  │── initialize(protocolVersion) ──────▶
  │◀── result(serverInfo, capabilities)─│
  │                                     │
  │── notifications/initialized ────────▶  (no response)
  │                                     │
  │── tools/list ───────────────────────▶
  │◀── result({tools: [...]}) ──────────│
  │                                     │
  │── tools/call({name, arguments}) ────▶
  │◀── result({content: [...]}) ────────│
```

The `initialize` exchange is the place where the two sides agree on a protocol version and where the server tells the client which optional capabilities (`tools`, `resources`, `prompts`, `sampling`, …) it implements. The notebook implements `tools` and `resources`.


## 3. JSON-RPC envelope in code

The full implementation lives in [`src/mcp_core/protocol.py`](../src/mcp_core/protocol.py). The two essentials are the `JsonRpcRequest` and `JsonRpcResponse` Pydantic models.


In [ ]:
from src.mcp_core.protocol import (
    JsonRpcRequest, JsonRpcResponse, JsonRpcError, MCPError,
    INVALID_PARAMS, METHOD_NOT_FOUND,
)

# Construct a request directly.
req = JsonRpcRequest(id=1, method="tools/list", params={})
print(req.model_dump_json(exclude_none=True))

# Notifications carry no id.
notif = JsonRpcRequest(method="notifications/initialized")
print("is_notification:", notif.is_notification)


**Schema invariants enforced by the model:**

- `result` and `error` are mutually exclusive in a response. The Pydantic validator rejects any object that sets both or neither.
- The `id` field is `int | str | None`. A `None` id classifies the message as a notification.
- Error codes between `-32099` and `-32000` are reserved for *implementation-defined* errors; MCP layers application-specific codes (e.g. `TOOL_NOT_FOUND = -32001`) into that band.


## 4. Defining a tool: schema is the contract

A tool is three things: a name, a JSON-Schema describing its inputs, and a Python callable. The schema is what an LLM reads to decide *whether* to call the tool; the callable is what runs when it does.


In [ ]:
from src.mcp_core.tool_registry import Tool, ToolRegistry

# A toy temperature-conversion tool.
def celsius_to_fahrenheit(celsius: float) -> str:
    return f"{(celsius * 9 / 5) + 32:.2f} F"

c2f = Tool(
    name="celsius_to_fahrenheit",
    description="Convert a temperature from Celsius to Fahrenheit.",
    input_schema={
        "type": "object",
        "properties": {
            "celsius": {"type": "number", "description": "Temperature in Celsius."}
        },
        "required": ["celsius"],
        "additionalProperties": False,
    },
    handler=celsius_to_fahrenheit,
)

print(c2f.invoke({"celsius": 100}))      # -> "212.00 F"


In [ ]:
# Schema validation fires *before* the handler runs.
try:
    c2f.invoke({"celsius": "not a number"})
except MCPError as exc:
    print("rejected:", exc.code, exc.message)


Why this matters: when an LLM emits an `Action Input` JSON object, we want it to fail loudly and recoverably when the JSON does not match the contract — not to crash the handler with a `TypeError`. The schema validator gives us a structured `INVALID_PARAMS` error that the agent can feed back to the model for self-correction.


## 5. Spinning up the server in-process

We won't run the stdio loop in the notebook (because that would block on stdin), but we can drive the server directly by passing JSON frames to `handle()`. This is exactly what the test suite does.


In [ ]:
import json
from src.mcp_core.server import MCPServer

registry = ToolRegistry()
registry.register(c2f)
server = MCPServer(name="weather-server", version="0.1.0", tools=registry)

def call(method, params=None, _id=[1]):
    req = {"jsonrpc": "2.0", "id": _id[0], "method": method, "params": params or {}}
    _id[0] += 1
    raw = server.handle(json.dumps(req))
    return json.loads(raw) if raw else None

print(call("initialize"))
print(call("tools/list"))
print(call("tools/call", {"name": "celsius_to_fahrenheit", "arguments": {"celsius": 0}}))


**Notice three things:**

1. The `initialize` response advertises which capabilities the server supports — populated dynamically from the registered tools and resources.
2. `tools/list` returns the descriptor *including the JSON-Schema*. An LLM can read this and reason about how to call the tool without seeing the Python source.
3. `tools/call` wraps the handler's return value inside a `content` array of typed blocks. This mirrors how MCP servers can return text, images, or binary blobs uniformly.


## 6. Resources: read-only data, addressable by URI

A *resource* is an artifact the client can read into context. The protocol does not distinguish a 100-page PDF from a 10-row database table; both are URI-addressable streams of bytes with a MIME type.

We implement two resource providers:

- **`FilesystemResources`** — files beneath a sandbox root, with a `commonpath` check that rejects symlink escapes.
- **`SQLiteResources`** — tables of a read-only SQLite database, each exposed as `sqlite://<table>`.


In [ ]:
import tempfile, os, sqlite3, shutil
from src.mcp_core.resources import FilesystemResources, SQLiteResources

# 1. Filesystem provider: create a temp directory with two files.
tmp_root = tempfile.mkdtemp()
with open(f"{tmp_root}/notes.md", "w") as f:
    f.write("# Project notes\n\nRAG works best when chunks match query length.")
os.makedirs(f"{tmp_root}/subdir", exist_ok=True)
with open(f"{tmp_root}/subdir/todo.txt", "w") as f:
    f.write("- write the MCP notebook\n- add tests\n")

fs = FilesystemResources(tmp_root)
for r in fs.list_resources():
    print(r)

# Read one back.
print("---")
print(fs.read("file://notes.md"))


In [ ]:
# 2. Sandbox escape check: ../ paths and symlinks are rejected.
from src.mcp_core.protocol import MCPError
try:
    fs.read("file://../../etc/passwd")
except MCPError as exc:
    print("blocked:", exc.message)

shutil.rmtree(tmp_root)


In [ ]:
# 3. SQLite provider: build a tiny DB and read a table through MCP.
db_path = tempfile.mktemp(suffix=".sqlite")
conn = sqlite3.connect(db_path)
conn.execute("CREATE TABLE papers (id TEXT, title TEXT)")
conn.executemany(
    "INSERT INTO papers VALUES (?, ?)",
    [("2212.10496", "Precise Zero-Shot Dense Retrieval (HyDE)"),
     ("1810.04805", "BERT: Pre-training of Deep Bidirectional Transformers")],
)
conn.commit(); conn.close()

db = SQLiteResources(db_path)
print(db.list_resources())
print(db.read("sqlite://papers"))


**Why this matters for agents.** The Week 6 capstone uses the SQLite resource provider to expose the arXiv papers it ingests, so the LLM can query the metadata table *as a resource* rather than going through an ad-hoc tool. Resources are the right abstraction for "look up these facts"; tools are the right abstraction for "do this action".


## 7. Driving the server from a client

The repository ships [`src/mcp_core/client.py`](../src/mcp_core/client.py), a minimal stdio client. To use it, you would launch the server as a subprocess:

```bash
python -m src.mcp_core.server --config configs/mcp_server.yaml
```

…and then connect:

```python
from src.mcp_core.client import MCPStdioClient
client = MCPStdioClient(["python", "-m", "src.mcp_core.server",
                        "--config", "configs/mcp_server.yaml"])
client.initialize()
print(client.list_tools())
print(client.call_tool("celsius_to_fahrenheit", {"celsius": 25}))
client.close()
```

The same wire protocol is what Claude Desktop speaks to your server. Any MCP-compliant client (the official SDKs, the in-IDE assistants, your own Python script) can call your tools and read your resources.


## 8. Exercises

1. **Add a writeable resource.** Extend `FilesystemResources` to support `resources/write` (you will need a new method handler on the server). Discuss what the security implications are — and why MCP intentionally puts a much harder bar on write than on read.
2. **Add a third capability.** Implement `prompts/list` and `prompts/get`, returning a small library of parameterized prompt templates. (Hint: a *prompt template* is just a tool whose handler does string substitution.)
3. **Negotiate protocol version.** Make `initialize` reject a client whose `protocolVersion` field is incompatible with the server's. What is the right *latticing* — semantic-version compatibility, or exact match?


## 9. Take-aways

- MCP is **JSON-RPC 2.0 with a fixed method set**. Once you can parse and dispatch JSON-RPC, MCP is a small extension.
- The **schema is the contract**. Validate inputs before invoking handlers, surface errors structurally, and your agents will recover gracefully.
- **Tools are for action, resources are for facts.** Read-only data should live behind `resources/read`, not inside a tool wrapper.
- The transport (stdio in our case) is **incidental**; the same server can be exposed over WebSocket or HTTP by swapping the I/O loop.

➡ Next week we use these foundations to build a hierarchical retrieval pipeline that an MCP-connected LLM can query.
